##[1단계] 환경 세팅 및 모델 로드

In [ ]:
import os
import random
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ==========================================
# 1. 재현성(Reproducibility)을 위한 시드 고정
# ==========================================
def seed_everything(seed=42):
    """
    실험 결과가 매번 달라지는 것을 방지하기 위해 랜덤 시드를 고정합니다.
    하이퍼파라미터 튜닝 시 성능 변화를 객관적으로 비교하기 위해 필수적인 작업입니다.
    """
    random.seed(seed)                            # 파이썬 내장 random 모듈 난수 고정
    os.environ['PYTHONHASHSEED'] = str(seed)     # 파이썬 딕셔너리/셋 구조의 해시 무작위성 고정
    np.random.seed(seed)                         # Numpy 배열 난수 고정
    torch.manual_seed(seed)                      # PyTorch CPU 난수 고정
    torch.cuda.manual_seed(seed)                 # PyTorch GPU 난수 고정
    torch.cuda.manual_seed_all(seed)             # 멀티 GPU 사용 시 필수
    torch.backends.cudnn.deterministic = True    # CuDNN 연산의 결정론적 보장
    torch.backends.cudnn.benchmark = False       # CuDNN 최적화 비활성화 (재현성 우선)

seed_everything(42)

# ==========================================
# 2. 디바이스 설정 (Colab T4 GPU 활용)
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ 현재 사용 중인 디바이스: {device}")


# ==========================================
# 3. 모델 및 토크나이저 로드 (BigBird: 최대 4096 토큰 지원)
# ==========================================

# 💡 삭제된 모델 대신 검증된 한국어 BigBird 모델로 교체
MODEL_NAME = "monologg/kobigbird-bert-base"

num_labels = 2 # 참(0), 거짓(1) 이진 분류

print(f"[{MODEL_NAME}] 토크나이저 및 모델 로딩 중...")

# 토크나이저 로드 (BigBird 전용 토크나이저를 자동으로 불러옵니다)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Sequence Classification 용도의 모델 로드
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    ignore_mismatched_sizes=True
)

# 모델을 GPU 메모리로 이동
model.to(device)

print("✅ 모델 및 토크나이저 로드 완료!")

#[1.5단계]Data Augmentation

In [ ]:
import pandas as pd
import random
import numpy as np

random.seed(42)
np.random.seed(42)

def random_deletion(words, p=0.15):
    if len(words) == 1: return words
    new_words = [word for word in words if random.uniform(0, 1) > p]
    if len(new_words) == 0: return [random.choice(words)]
    return new_words

def random_swap(words, n=1):
    new_words = words.copy()
    for _ in range(n):
        if len(new_words) >= 2:
            idx1, idx2 = random.sample(range(len(new_words)), 2)
            new_words[idx1], new_words[idx2] = new_words[idx2], new_words[idx1]
    return new_words

def augment_text(text):
    # 🛡️ 결측치(NaN) 방어 로직: 문자열이 아니면 그대로 반환
    if not isinstance(text, str):
        return str(text) if text else " "

    words = text.split()
    if len(words) < 3: return text

    if random.random() > 0.5:
        augmented_words = random_deletion(words, p=0.2)
    else:
        augmented_words = random_swap(words, n=1)

    return " ".join(augmented_words)

# ==========================================
# 2. 원본 데이터 로드 및 1:1 비율 유지 증강
# ==========================================
ORIGINAL_CSV = "nli_train_data_10k_v2.csv"
AUGMENTED_CSV = "nli_train_data_augmented.csv"

print(f"[{ORIGINAL_CSV}] 데이터를 읽어옵니다...")
df = pd.read_csv(ORIGINAL_CSV)

# 🛡️ 비율 유지 로직: 추가가 아닌 '교체(Replace)'를 위해 인덱스 분리
df_true_all = df[df['label'] == 0]
df_true_to_augment = df_true_all.sample(n=2000, random_state=42).copy()

print("문장 변형(EDA) 적용 중... (어휘적 과적합 파괴)")
df_true_to_augment['hypothesis'] = df_true_to_augment['hypothesis'].apply(augment_text)

# 원본 데이터 프레임에서 증강할 대상이었던 원본 2000개를 삭제
df_remaining = df.drop(df_true_to_augment.index)

# 남은 데이터 8000개 + 변형된 데이터 2000개 결합 (총 10,000개, 5000:5000 완벽 유지)
df_augmented = pd.concat([df_remaining, df_true_to_augment], ignore_index=True)

# 데이터 셔플
df_augmented = df_augmented.sample(frac=1, random_state=42).reset_index(drop=True)

df_augmented.to_csv(AUGMENTED_CSV, index=False)

print("\n" + "="*60)
print(f"✅ 데이터 교체 증강 완료! 총 {len(df_augmented)}개의 데이터가 [{AUGMENTED_CSV}]로 저장되었습니다.")
print("   (클래스 불균형 방지: Label 0과 1의 1:1 비율 유지 완료)")
print("="*60)

##[2단계]데이터셋 구축 및 DataLoader 생성

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import DataCollatorWithPadding

# ==========================================
# 1. Custom Dataset 클래스 정의 (NLI 전용)
# ==========================================
class NLIDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=512): # max_length 기본값을 512로 변경 권장
        # 결측치가 있으면 에러가 나므로 미리 제거
        self.df = df.dropna(subset=['premise', 'hypothesis', 'label']).reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        premise = str(row['premise'])
        hypothesis = str(row['hypothesis'])
        label = int(row['label'])

        # 동적 패딩(Dynamic Padding)을 위해 padding 옵션 제외
        encoding = self.tokenizer(
            premise,
            hypothesis,
            truncation=True,        # max_length보다 길면 자름
            max_length=self.max_length,
            # return_tensors='pt' 삭제
        )

        # 💡 [수정] torch.tensor로 수동 변환하지 마세요!
        # DataCollatorWithPadding이 리스트를 받아 패딩 후 텐서로 변환해 줍니다.
        item = {key: val for key, val in encoding.items()}
        item['labels'] = label # 라벨도 정수 그대로 넘깁니다.

        return item

# ==========================================
# 2. DataLoader 생성 함수
# ==========================================
def create_data_loaders(csv_path, tokenizer, batch_size=16, max_length=512):
    print(f"[{csv_path}] Data Pre-Processing Start")

    # 1. CSV 파일 읽기
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"🚨 에러: '{csv_path}' 파일을 찾을 수 없습니다. 경로를 확인하세요.")
        return None, None

    # 2. Train Data = 80% / Validation 20%
    train_df, valid_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
    print(f"✅ 데이터 분할 완료 -> Train:{len(train_df)}개, Valid:{len(valid_df)}개")

    # 3. Dataset 인스턴스화
    train_dataset = NLIDataset(train_df, tokenizer, max_length)
    valid_dataset = NLIDataset(valid_df, tokenizer, max_length)

    # 4. Data Collator 선언
    # 배치(Batch)별로 가장 긴 텍스트의 길이에 맞춰 동적으로 패딩(Padding)을 추가합니다.
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # 5. DataLoader로 래핑
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        # Windows 환경에서는 num_workers > 0 일 때 에러가 날 수 있으니 주의. Colab은 문제없음.
        num_workers=2,
        pin_memory=True,
        collate_fn=data_collator # 동적 패딩 적용
    )

    valid_loader = DataLoader(
        valid_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        collate_fn=data_collator # 동적 패딩 적용
    )

    print(f"✅ DataLoader 준비 완료! (Train 배치 수: {len(train_loader)}, Valid 배치 수: {len(valid_loader)})")
    return train_loader, valid_loader

# ==========================================
# 3. 실행 예시
# ==========================================

train_loader, valid_loader = create_data_loaders(
    csv_path="nli_train_data_10k_v2.csv",
    tokenizer=tokenizer,
    batch_size=4,           # 🚨 (중요) 메모리 폭발 방지! 절대 4를 넘기지 마세요.
    max_length=1024         # 기사 전문 100% 반영 (최대 길이 853 커버)
)

##[3단계]학습 루프 및 Optimizer/Scheduler 설정

In [ ]:
import os
import torch
import numpy as np
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from torch.amp import autocast, GradScaler # 💡 [수석 최적화] PyTorch 최신 API로 경고(Warning) 제거
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, recall_score, f1_score

# ==========================================
# 1. 하이퍼파라미터 세팅
# ==========================================
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
ACCUMULATION_STEPS = 8
WARMUP_RATIO = 0.1
SAVE_DIR = "./saved_model"

if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

print("✅ 옵티마이저 및 스케줄러 설정 중...")

# ==========================================
# 2. 옵티마이저 & 스케줄러 설정
# ==========================================
no_decay = ['bias', 'LayerNorm.weight']
optimizer_grouped_parameters = [
    {
        'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
        'weight_decay': WEIGHT_DECAY
    },
    {
        'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
        'weight_decay': 0.0
    }
]

optimizer = AdamW(optimizer_grouped_parameters, lr=LEARNING_RATE)

total_steps = (len(train_loader) // ACCUMULATION_STEPS) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

# 💡 [수석 최적화] 'cuda' 디바이스를 명시하는 최신 스케일러 문법
scaler = GradScaler('cuda')

# ==========================================
# 3. Train & Validation Loop (완전한 학습 루프)
# ==========================================
print(f"▶ 총 예상 최적화 스텝 수: {total_steps} (Warmup: {warmup_steps} 스텝)")
print("🚀 본격적인 학습을 시작합니다!")

best_recall = 0.0

for epoch in range(EPOCHS):
    # ---------------------------------------
    # [Train Phase]
    # ---------------------------------------
    model.train()
    total_train_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")

    for step, batch in enumerate(progress_bar):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # 💡 [추가] BigBird(BERT 계열)를 위한 문장 구분 텐서 추출
        token_type_ids = batch['token_type_ids'].to(device) if 'token_type_ids' in batch else None

        # 🛡️ [강력한 방어 코드]: device-side assert 원천 차단
        vocab_size = model.config.vocab_size
        input_ids = torch.clamp(input_ids, min=0, max=vocab_size - 1)
        labels = torch.clamp(labels, min=0, max=1)

        # 💡 [수석 최적화] 최신 autocast API 적용
        with autocast('cuda'):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,   # 💡 [추가] 모델에 토큰 타입 전달
                labels=labels
            )
            loss = outputs.loss
            loss = loss / ACCUMULATION_STEPS

        scaler.scale(loss).backward()
        total_train_loss += loss.item() * ACCUMULATION_STEPS

        if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        progress_bar.set_postfix({'loss': f"{loss.item() * ACCUMULATION_STEPS:.4f}"})

    avg_train_loss = total_train_loss / len(train_loader)

    # ---------------------------------------
    # [Validation Phase]
    # ---------------------------------------
    model.eval()
    total_valid_loss = 0
    all_preds = []
    all_labels = []

    valid_bar = tqdm(valid_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Valid]")

    # 💡 [해결] 들여쓰기(Indentation) 라인 완벽 정렬 완료
    with torch.no_grad():
        for batch in valid_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # 💡 검증 단계에서도 문장 구분 텐서 추출
            token_type_ids = batch['token_type_ids'].to(device) if 'token_type_ids' in batch else None

            # 🛡️ 검증 단계에서도 동일한 방어 코드 적용
            input_ids = torch.clamp(input_ids, min=0, max=vocab_size - 1)
            labels = torch.clamp(labels, min=0, max=1)

            with autocast('cuda'):
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_type_ids=token_type_ids,  # 💡 모델에 토큰 타입 전달
                    labels=labels
                )
                loss = outputs.loss
                logits = outputs.logits

            total_valid_loss += loss.item()

            preds = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            valid_bar.set_postfix({'val_loss': f"{loss.item():.4f}"})

    avg_valid_loss = total_valid_loss / len(valid_loader)
    acc = accuracy_score(all_labels, all_preds)

    rec = recall_score(all_labels, all_preds, pos_label=1, zero_division=0)
    f1 = f1_score(all_labels, all_preds, pos_label=1, zero_division=0)

    print(f"\n📊 Epoch {epoch+1} Results:")
    print(f"   Train Loss: {avg_train_loss:.4f} | Valid Loss: {avg_valid_loss:.4f}")
    print(f"   Accuracy: {acc:.4f} | ★Recall: {rec:.4f} | F1-Score: {f1:.4f}")

    # ---------------------------------------
    # [Model Checkpoint] 최고 성능 모델 저장
    # ---------------------------------------
    if rec > best_recall:
        print(f"🎉 Recall 성능이 개선되었습니다! ({best_recall:.4f} -> {rec:.4f}). 모델을 저장합니다.")
        best_recall = rec

        model.save_pretrained(SAVE_DIR)
        tokenizer.save_pretrained(SAVE_DIR)
    print("-" * 60)

# Local PC Zip File Download

In [ ]:
import shutil
from google.colab import files

# 1. 저장된 모델 폴더 경로 (예: './saved_model')
model_folder_path = './saved_model'
zip_file_name = 'my_trained_model'

print("📦 모델 폴더를 압축하는 중입니다...")
# 폴더를 ZIP 파일로 압축
shutil.make_archive(zip_file_name, 'zip', model_folder_path)

print("📥 압축 완료! 로컬 PC로 다운로드를 시작합니다.")
# 브라우저를 통해 내 PC로 다운로드
files.download(f"{zip_file_name}.zip")